# ULTRON Training Control

Portable browser launcher for Kaggle, Google Colab, Lightning AI and GitHub Actions. This notebook downloads the shared Python engine, random-initialized Transformer, deterministic mathematics generator, independent verifier, safe checkpoint manager and settings from the ULTRONAI GitHub repository.

The notebook checks runtime detection, exact model size, independently verified math examples and a local checkpoint save/restore cycle. It does not start a training run yet.

On Kaggle, enable Internet in Notebook Settings so the notebook can retrieve the project files from GitHub. No secrets are stored in this notebook.

In [ ]:
import json
import pathlib
import subprocess
import sys
import urllib.parse
import urllib.request

REPOSITORY = "shayaanfarsheed-cmd/ULTRONAI"
BRANCH = "main"
BASE_URL = f"https://raw.githubusercontent.com/{REPOSITORY}/{BRANCH}"
PROJECT_DIR = pathlib.Path.cwd() / ".ultron_training_control"
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
FILES = ("train.py", "ultron_model.py", "ultron_data.py", "ultron_verify.py", "ultron_checkpoint.py", "config.json", "servers.json", "requirements.txt")

for filename in FILES:
    request = urllib.request.Request(
        f"{BASE_URL}/{filename}",
        headers={"User-Agent": "ULTRON-Training-Control"},
    )
    with urllib.request.urlopen(request, timeout=45) as response:
        final_host = urllib.parse.urlparse(response.geturl()).hostname
        if final_host != "raw.githubusercontent.com":
            raise RuntimeError("Unexpected download host; refusing to save the project file.")
        content = response.read(2_000_001)
    if len(content) > 2_000_000:
        raise RuntimeError(f"Project file is unexpectedly large: {filename}")
    if filename.endswith(".json"):
        json.loads(content.decode("utf-8"))
    (PROJECT_DIR / filename).write_bytes(content)

print("ULTRON project files retrieved from GitHub.")
print("Project folder:", PROJECT_DIR)

In [ ]:
command = [
    sys.executable,
    str(PROJECT_DIR / "train.py"),
    "--preflight",
    "--config",
    str(PROJECT_DIR / "config.json"),
    "--servers",
    str(PROJECT_DIR / "servers.json"),
]
result = subprocess.run(command, cwd=PROJECT_DIR, check=False)
if result.returncode != 0:
    raise RuntimeError(f"ULTRON preflight failed with exit code {result.returncode}")

summary_command = command + ["--model-summary"]
summary_result = subprocess.run(summary_command, cwd=PROJECT_DIR, check=False)
if summary_result.returncode != 0:
    raise RuntimeError(f"ULTRON model summary failed with exit code {summary_result.returncode}")

data_command = command + ["--data-smoke-test"]
data_result = subprocess.run(data_command, cwd=PROJECT_DIR, check=False)
if data_result.returncode != 0:
    raise RuntimeError(f"ULTRON data verification failed with exit code {data_result.returncode}")

checkpoint_command = command + ["--checkpoint-smoke-test"]
checkpoint_result = subprocess.run(checkpoint_command, cwd=PROJECT_DIR, check=False)
if checkpoint_result.returncode != 0:
    raise RuntimeError(f"ULTRON checkpoint smoke test failed with exit code {checkpoint_result.returncode}")